In [2]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "Movie+Assignment+Data.csv" 
movies_df = pd.read_csv(file_path)


movies_df = movies_df.loc[:, ~movies_df.columns.str.contains('^Unnamed')]


movies_df.columns = movies_df.columns.str.strip()


movies_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct") for col in movies_df.columns]


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "movies_data"
columns = movies_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()
print("Table recreated with schema:")
print(create_stmt)


columns_list = list(movies_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

for _, row in movies_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row[columns_list]]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)
cur.close()
conn.close()

print("Data fetched from DB:")
print(df_from_db.head())


Table recreated with schema:

CREATE TABLE "movies_data" (
  "color" TEXT,
  "director_name" TEXT,
  "num_critic_for_reviews" FLOAT,
  "duration" FLOAT,
  "director_facebook_likes" FLOAT,
  "actor_3_facebook_likes" FLOAT,
  "actor_2_name" TEXT,
  "actor_1_facebook_likes" FLOAT,
  "gross" FLOAT,
  "genres" TEXT,
  "actor_1_name" TEXT,
  "movie_title" TEXT,
  "num_voted_users" INT,
  "cast_total_facebook_likes" INT,
  "actor_3_name" TEXT,
  "facenumber_in_poster" FLOAT,
  "plot_keywords" TEXT,
  "movie_imdb_link" TEXT,
  "num_user_for_reviews" FLOAT,
  "language" TEXT,
  "country" TEXT,
  "content_rating" TEXT,
  "budget" FLOAT,
  "title_year" FLOAT,
  "actor_2_facebook_likes" FLOAT,
  "imdb_score" FLOAT,
  "aspect_ratio" FLOAT,
  "movie_facebook_likes" INT
);

Data inserted successfully
Data fetched from DB:
   color      director_name  num_critic_for_reviews  duration  \
0  Color      James Cameron                   723.0     178.0   
1  Color     Gore Verbinski                   302.0